In [1]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers=["127.0.0.1:9092"],
    api_version=(2, 8, 0),
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

print("Connected!")

Connected!


In [2]:
producer = KafkaProducer(
    bootstrap_servers=["127.0.0.1:9092"],
    api_version=(2, 8, 0),

    value_serializer=lambda value: json.dumps(
        value,
        ensure_ascii=False
    ).encode("utf-8"),

    key_serializer=lambda key: key.encode("utf-8"),

    acks="all",
    retries=5,
    batch_size=16384,
    linger_ms=10
)

In [ ]:
import json
import subprocess
import time
from pathlib import Path
from decimal import Decimal
from datetime import datetime, date

import ijson
from kafka import KafkaProducer
from kafka.errors import KafkaError


# ============================================================
# Configuration
# ============================================================

HDFS_COMMAND = Path(r"C:\Hadoop\bin\hdfs.cmd")

HDFS_SESSION_FILE = "/ecommerce/raw/sessions/sessions_0.json"

KAFKA_BOOTSTRAP_SERVERS = ["127.0.0.1:9092"]
KAFKA_TOPIC = "ecommerce_sessions"

# Explicit API version (avoids auto-detection timeout)
KAFKA_API_VERSION = (2, 8, 0)

# Number of messages to send
# Set to None to stream the whole file
MAX_MESSAGES = 1000

# Delay between records
INTERVAL_SECONDS = 0.5


# ============================================================
# Custom JSON Serializer
# ============================================================

def json_serializer(obj):
    """
    Convert unsupported Python objects into JSON serializable types.
    """

    if isinstance(obj, Decimal):
        return float(obj)

    if isinstance(obj, (datetime, date)):
        return obj.isoformat()

    raise TypeError(
        f"Object of type {type(obj).__name__} is not JSON serializable"
    )


# ============================================================
# Validation
# ============================================================

if not HDFS_COMMAND.exists():
    raise FileNotFoundError(
        f"hdfs.cmd was not found at: {HDFS_COMMAND}"
    )


# ============================================================
# Kafka Producer
# ============================================================

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,

    api_version=KAFKA_API_VERSION,

    value_serializer=lambda value: json.dumps(
        value,
        ensure_ascii=False,
        default=json_serializer
    ).encode("utf-8"),

    key_serializer=lambda key: key.encode("utf-8"),

    acks="all",

    retries=5,

    batch_size=16384,

    linger_ms=10
)

print("Connected to Kafka successfully.")


# ============================================================
# Open HDFS File
# ============================================================

command = [
    str(HDFS_COMMAND),
    "dfs",
    "-cat",
    HDFS_SESSION_FILE
]

print(f"Opening HDFS file: {HDFS_SESSION_FILE}")

hdfs_process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1
)

if hdfs_process.stdout is None:
    raise RuntimeError(
        "Could not open the HDFS output stream."
    )


# ============================================================
# Stream Sessions to Kafka
# ============================================================

sent_count = 0

try:

    sessions = ijson.items(
        hdfs_process.stdout,
        "item"
    )

    for session in sessions:

        session_id = session.get("session_id")

        if not session_id:
            print("Skipping record without session_id")
            continue

        future = producer.send(
            topic=KAFKA_TOPIC,
            key=session_id,
            value=session
        )

        try:

            metadata = future.get(timeout=30)

            sent_count += 1

            print(
                f"Sent {sent_count}: "
                f"session_id={session_id}, "
                f"partition={metadata.partition}, "
                f"offset={metadata.offset}"
            )

        except KafkaError as error:

            print(
                f"Failed to send session "
                f"{session_id}: {error}"
            )

        if (
            MAX_MESSAGES is not None
            and sent_count >= MAX_MESSAGES
        ):
            break

        time.sleep(INTERVAL_SECONDS)

except Exception as error:

    print(f"\nStreaming error: {error}")

finally:

    print("\nClosing producer...")

    producer.flush()
    producer.close()

    if hdfs_process.stdout:
        hdfs_process.stdout.close()

    hdfs_process.terminate()

    stderr_output = ""

    if hdfs_process.stderr:
        stderr_output = hdfs_process.stderr.read()
        hdfs_process.stderr.close()

    if stderr_output.strip():
        print("\nHDFS messages:")
        print(stderr_output)

print(f"\nProducer finished. Messages sent: {sent_count}")

Connected to Kafka successfully.
Opening HDFS file: /ecommerce/raw/sessions/sessions_0.json
Sent 1: session_id=sess_3a1aabf936, partition=1, offset=337
Sent 2: session_id=sess_9aacfbc27a, partition=1, offset=338
Sent 3: session_id=sess_967eb4e750, partition=1, offset=339
Sent 4: session_id=sess_42a4ffd18b, partition=2, offset=328
Sent 5: session_id=sess_07e4148061, partition=2, offset=329
Sent 6: session_id=sess_ff88177e5f, partition=2, offset=330
Sent 7: session_id=sess_4aa577c5ca, partition=1, offset=340
Sent 8: session_id=sess_a270ffdd68, partition=1, offset=341
Sent 9: session_id=sess_e7e47e5e27, partition=1, offset=342
Sent 10: session_id=sess_7e1410a164, partition=0, offset=337
Sent 11: session_id=sess_83432fbf5f, partition=1, offset=343
Sent 12: session_id=sess_aa524f7692, partition=1, offset=344
Sent 13: session_id=sess_ef5422c10b, partition=1, offset=345
Sent 14: session_id=sess_9bb7dd9cdf, partition=1, offset=346
Sent 15: session_id=sess_5c02014227, partition=1, offset=347
Se

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EcommerceKafkaStreaming")
    .master("local[2]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8"
    )
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)